In [39]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [40]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [41]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

# Q01 How many lesson pages are in the dataset?

In [42]:
len(documents)

72

# Q02. What's the filename of the first result?

In [43]:
question='How does the agentic loop keep calling the model until it stops?'

In [44]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=["filename"]
)

index.fit(documents)

In [45]:
index.search(question)[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

# Q03. Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

In [46]:
from hw_helper import RAGBase

In [47]:
question = "How does the agentic loop keep calling the model until it stops?"

In [48]:
assistant = RAGBase(index,openai_client)

In [49]:
ans= assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [50]:
ans

{'response': 'The loop keeps calling the model in a `while True` loop and checks each response for any `function_call` items.\n\n- If the response includes a function call, the code runs the tool, appends the tool output to `messages`, and continues.\n- If the response has no function calls, `has_function_calls` stays `False`, and the loop `break`s.\n\nSo the stop condition is: **no function calls in the model’s response**.',
 'input_tokens': 7126,
 'output_tokens': 98,
 'total_tokens': 7224}

input prompt tokens sent are 7126

# Q04. How many chunks do you get?

In [51]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [52]:
len(chunks)

295

# Q05. Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

In [53]:
# for documents
index = Index(
    text_fields=['content'],
    keyword_fields=["filename"]
)

index.fit(documents)

# for chunks
index_chunks = Index(
    text_fields=['content'],
    keyword_fields=["filename"]
)

index_chunks.fit(chunks)

In [54]:
# for documents
assistant = RAGBase(index,openai_client)

# for chunks
assistant_chunks = RAGBase(index_chunks,openai_client)

In [55]:
answer_doc = assistant.rag("How does the agentic loop keep calling the model until it stops?")
answer_chunks = assistant_chunks.rag("How does the agentic loop keep calling the model until it stops?")

In [56]:
answer_doc['input_tokens']

7126

In [57]:
final_answer = answer_doc['input_tokens']/answer_chunks['input_tokens']
print(final_answer)

# 3 * fewer input tokens does the chunked version send.

3.086184495452577


# Q06 How many times did the agent call search?

In [58]:
question = 'How does the agentic loop work, and how is it different from plain RAG?'

In [59]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [60]:
index_chunks = Index(
    text_fields=['content'],
    keyword_fields=["filename"]
)

index_chunks.fit(chunks)

In [61]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [62]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

''

In [67]:
# letting the agent to use the search tool to answer questions about the course

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index_chunks.search(
        query,
        num_results=5,
    )

In [64]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [65]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join now eligibility registration"}', call_id='call_Q8uoKv9ge8II4cFn7rdxzJOG', name='search', type='function_call', id='fc_0a068687e6dadc30006a41748bb8cc8192bab340a516a8c77e', namespace=None, status='completed')]

In [68]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [69]:
# sending the result back to the model
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [70]:
# Asking the model again
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nAccording to the FAQ, if you want to receive a certificate, you need to submit your project while submissions are still open.'

In [71]:
# a function call helper
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [72]:
# processing the question with the agentic loop

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late registration"}


In [73]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [76]:
agent_loop(instructions, "How does the agentic loop work, and how is it different from plain RAG")

iteration #1...


function_call: search {"query":"agentic loop plain RAG difference"}
function_call: search {"query":"agentic loop how it works RAG"}
function_call: search {"query":"agentic loop retrieval augmented generation agentic"}
iteration #2...
ASSISTANT:
The agentic loop is a **repeated cycle** where the model can choose tools, not just answer once.

From the course FAQ/lessons:

- An agentic loop keeps **calling the model**
- If the model returns a **function/tool call**, your code executes it
- The tool result is added back to the **message history**
- The loop repeats until the model returns a **final answer with no more tool calls**

So the basic flow is:

1. Send the user question + prior messages to the LLM
2. LLM decides whether to call a tool
3. If it does, your code runs the tool
4. Add the tool output back into memory
5. Repeat until no function calls remain

The lesson says this is the core idea: the model is in the driver’s seat, and the loop continues until it stops asking for tools

'The agentic loop is a **repeated cycle** where the model can choose tools, not just answer once.\n\nFrom the course FAQ/lessons:\n\n- An agentic loop keeps **calling the model**\n- If the model returns a **function/tool call**, your code executes it\n- The tool result is added back to the **message history**\n- The loop repeats until the model returns a **final answer with no more tool calls**\n\nSo the basic flow is:\n\n1. Send the user question + prior messages to the LLM\n2. LLM decides whether to call a tool\n3. If it does, your code runs the tool\n4. Add the tool output back into memory\n5. Repeat until no function calls remain\n\nThe lesson says this is the core idea: the model is in the driver’s seat, and the loop continues until it stops asking for tools.\n\nHow it differs from plain RAG:\n\n- **Plain RAG** is a fixed pipeline:\n  - `search(question) -> build_prompt(question, search_results) -> llm(prompt)`\n- The search happens **once**\n- The retrieved context is then used t

#search has been call 3 times